In [ ]:
!pip install openpyxl -q
print("✅ Dependencias listas.")

In [ ]:
from google.colab import auth
auth.authenticate_user()                 # pide permiso con tu cuenta
from googleapiclient.discovery import build
drive_service = build('drive', 'v3')
print("✅ Conectado a la API de Google Drive.")

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# 👉 PEGA AQUÍ el ID de tu carpeta A (lo sacas de la URL de la carpeta en Drive,
#    la parte después de /folders/ ... ejemplo: 1A2b3C4d5E6f7G8h9I0j)
ID_CARPETA_A = "PON_AQUI_EL_ID_DE_LA_CARPETA_A"

SHEET_MIME  = 'application/vnd.google-apps.spreadsheet'
FOLDER_MIME = 'application/vnd.google-apps.folder'

def listar_carpeta(folder_id):
    """Devuelve los elementos directos de una carpeta (por su ID)."""
    q = f"'{folder_id}' in parents and trashed=false"
    res = drive_service.files().list(
        q=q, fields="files(id,name,mimeType)", pageSize=1000,
        supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
    return res.get('files', [])

def listar_recursivo(folder_id, prefijo=''):
    """Recorre A y sus subcarpetas (B, C, ...) y lista TODOS los archivos."""
    items = []
    for f in listar_carpeta(folder_id):
        if f['mimeType'] == FOLDER_MIME:
            items += listar_recursivo(f['id'], prefijo + f['name'] + '/')
        else:
            items.append((prefijo + f['name'], f['id'], f['mimeType']))
    return items

todos = listar_recursivo(ID_CARPETA_A)

# Filtramos: Excel (.xlsx/.xls o Google Sheet) y planos (.txt/.csv/.dat)
op_excel = [(n, (fid, mime)) for (n, fid, mime) in todos
            if n.lower().endswith(('.xlsx', '.xls')) or mime == SHEET_MIME]
op_txt   = [(n, (fid, mime)) for (n, fid, mime) in todos
            if n.lower().endswith(('.txt', '.csv', '.dat'))]

dd_excel = widgets.Dropdown(options=op_excel, description='EXCEL:',
                            layout=widgets.Layout(width='95%'))
dd_txt   = widgets.Dropdown(options=op_txt, description='TXT:',
                            layout=widgets.Layout(width='95%'))

print(f"Encontrados dentro de la carpeta A: {len(op_excel)} Excel y {len(op_txt)} planos.")
print("Selecciona en cada menú el archivo que quieres usar:")
display(widgets.VBox([dd_excel, dd_txt]))

In [ ]:
import io
from googleapiclient.http import MediaIoBaseDownload

def descargar(file_id, mime, destino):
    """Baja el archivo a la máquina de Colab. Si es Google Sheet, lo exporta a .xlsx."""
    if mime == SHEET_MIME:
        req = drive_service.files().export_media(
            fileId=file_id,
            mimeType='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')
    else:
        req = drive_service.files().get_media(fileId=file_id)
    with io.FileIO(destino, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, req)
        done = False
        while not done:
            _s, done = downloader.next_chunk()

excel_id, excel_mime = dd_excel.value
txt_id,   txt_mime   = dd_txt.value

RUTA_EXCEL = '/content/cuentas.xlsx'
RUTA_TXT   = '/content/archivo_plano.txt'

descargar(excel_id, excel_mime, RUTA_EXCEL)
descargar(txt_id,   txt_mime,   RUTA_TXT)
print("✅ Archivos descargados a Colab para procesar.")

In [ ]:
import pandas as pd

df_excel = pd.read_excel(RUTA_EXCEL, header=None, dtype=str)
serie = df_excel.iloc[1:, 3]                       # columna D, desde fila 2
serie = serie.dropna().astype(str).str.strip()
serie = serie[serie != '']
serie = serie.str.replace(r'\.0$', '', regex=True) # 123.0 -> 123

cuentas_lista = serie.tolist()
cuentas_set   = set(cuentas_lista)
print(f"✅ Cuentas leídas: {len(cuentas_lista):,} | únicas: {len(cuentas_set):,}")
print("Ejemplos:", cuentas_lista[:5])

In [ ]:
import re

ENCODING = 'utf-8'              # 👉 cambia a 'latin-1' si ves acentos raros
patron = re.compile(r'\d+')     # 👉 usa r'[A-Za-z0-9]+' si las cuentas llevan letras

resultados, cuentas_encontradas = [], set()
contador = 0
with open(RUTA_TXT, 'r', encoding=ENCODING, errors='replace') as f:
    for linea in f:
        contador += 1
        linea = linea.rstrip('\n').rstrip('\r')
        comunes = set(patron.findall(linea)) & cuentas_set
        if comunes:
            for cuenta in comunes:
                resultados.append((cuenta, linea))
                cuentas_encontradas.add(cuenta)
        if contador % 200000 == 0:
            print(f"   ... {contador:,} líneas | {len(resultados):,} coincidencias")
print(f"✅ Líneas: {contador:,} | Registros hallados: {len(resultados):,}")

In [ ]:
df_resultados = pd.DataFrame(resultados,
                columns=['Cuenta encontrada', 'Registro completo encontrado'])
no_encontradas = sorted(cuentas_set - cuentas_encontradas)
df_no = pd.DataFrame({'Cuenta no encontrada': no_encontradas})
print(f"Filas en el resultado: {len(df_resultados):,}")
df_resultados.head(10)

In [ ]:
import os
from datetime import datetime
from googleapiclient.http import MediaFileUpload

def obtener_o_crear_subcarpeta(nombre, parent_id):
    """Busca la subcarpeta dentro de A; si no existe, la crea."""
    q = (f"'{parent_id}' in parents and name='{nombre}' "
         f"and mimeType='{FOLDER_MIME}' and trashed=false")
    res = drive_service.files().list(q=q, fields='files(id,name)',
          supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
    if res.get('files'):
        return res['files'][0]['id']
    meta = {'name': nombre, 'mimeType': FOLDER_MIME, 'parents': [parent_id]}
    return drive_service.files().create(body=meta, fields='id',
           supportsAllDrives=True).execute()['id']

def subir(ruta_local, nombre, parent_id):
    meta = {'name': nombre, 'parents': [parent_id]}
    media = MediaFileUpload(ruta_local, resumable=True)
    return drive_service.files().create(body=meta, media_body=media,
           fields='id', supportsAllDrives=True).execute()['id']

# 👉 PUEDES MODIFICAR el nombre de la subcarpeta de resultados
NOMBRE_SUBCARPETA = 'Resultados_Busqueda'
id_resultados = obtener_o_crear_subcarpeta(NOMBRE_SUBCARPETA, ID_CARPETA_A)

marca = datetime.now().strftime('%Y%m%d_%H%M%S')
local_xlsx = f'/content/RESULTADO_BUSQUEDA_{marca}.xlsx'
local_txt  = f'/content/RESULTADO_BUSQUEDA_{marca}.txt'

LIMITE_EXCEL = 1_048_575
with pd.ExcelWriter(local_xlsx, engine='openpyxl') as writer:
    df_resultados.head(LIMITE_EXCEL).to_excel(writer, sheet_name='Encontrados', index=False)
    df_no.to_excel(writer, sheet_name='No_encontradas', index=False)

with open(local_txt, 'w', encoding='utf-8') as f:
    for _c, linea in resultados:
        f.write(linea + '\n')

subir(local_xlsx, os.path.basename(local_xlsx), id_resultados)
subir(local_txt,  os.path.basename(local_txt),  id_resultados)
print(f"✅ Resultados subidos a la subcarpeta '{NOMBRE_SUBCARPETA}' dentro de la carpeta A.")

In [ ]:
print("="*52)
print("           RESUMEN DE LA BÚSQUEDA")
print("="*52)
print(f" Cuentas leídas (filas)  : {len(cuentas_lista):,}")
print(f" Cuentas únicas a buscar : {len(cuentas_set):,}")
print(f" Cuentas ENCONTRADAS     : {len(cuentas_encontradas):,}")
print(f" Cuentas NO encontradas  : {len(cuentas_set) - len(cuentas_encontradas):,}")
print(f" Registros hallados      : {len(resultados):,}")
print(f" Subcarpeta de resultados: '{NOMBRE_SUBCARPETA}' (dentro de la carpeta A)")
print("="*52)